## 0. Setup (run first in every notebook)
Attach these inputs via **Add Input**: your uploaded `causal-rag-kaggle` dataset (this project) and, depending on the stage,
the Lending Club dataset (`wordsforthewise/lending-club`) and/or the artifacts dataset saved from the previous notebook.
Turn **Internet ON** (policy documents, models, RCT data) and set the accelerator to **GPU T4 x2** for stages 04-08.

In [ ]:
!pip -q install causal-learn econml sentence-transformers bitsandbytes accelerate gradio pypdf beautifulsoup4 scikit-uplift pyarrow 2>&1 | tail -1
import sys, os, glob
hits = glob.glob('/kaggle/input/**/causal_rag/__init__.py', recursive=True)
print('causal_rag copies found:'); [print('  ', h) for h in hits]
need = ('data_prep.py', 'causal_graph.py', 'retrieval.py', 'counterfactual.py', 'audit.py', 'compare.py', 'pipeline.py', 'live_demo.py')
complete = [h for h in hits if all(os.path.exists(os.path.join(os.path.dirname(h), n)) for n in need)]
v11 = [h for h in complete if 'prompt_graded' in open(os.path.join(os.path.dirname(h), 'prompts.py')).read()]
if not complete:
    for h in hits: print('FILES in', os.path.dirname(h), '->', sorted(os.listdir(os.path.dirname(h))))
assert complete, "No COMPLETE causal_rag package found in the attached dataset: re-upload causal_rag_kaggle.zip as a new dataset version."
assert v11, "The mounted package is an OLD version (prompts.py has no prompt_graded; need code 1.3.0). Upload the latest causal_rag_kaggle.zip as a NEW VERSION of the dataset, STOP the session (power icon), re-attach the latest version, and start again."
pkg = v11[0]
for m in [m for m in sys.modules if m.startswith('causal_rag')]: del sys.modules[m]   # drop any stale import
sys.path.insert(0, os.path.dirname(os.path.dirname(pkg)))
print('USING:', pkg)
from causal_rag import paths
paths.restore_artifacts()          # pulls artifacts from any attached output-dataset of an earlier stage
SMOKE   = False                    # True = tiny REAL-data dry run (minutes) to verify the pipeline
N_EVAL  = 12 if SMOKE else 600     # applicants scored by the LLM systems (v1.0/v1.1 used 300; ~5 h at 600, ~6.5 h at 800)
CHUNK   = 6  if SMOKE else 100     # LLM stage saves every CHUNK applicants and resumes after a crash
RUN_NAME = 'v1_2_run'               # name of this run's folder in the exported results
AUDIT_SYSTEM = 'causal_rag_smooth'  # system used by the audit agent: 'causal_rag' (v1.1 prompt) or 'causal_rag_smooth'
EPOCHS  = 3  if SMOKE else 40
N_REPORT= 3  if SMOKE else 10      # applicants that get full DAG + HTML + rationale
WITH_ECONML = False                # EconML causal forest is very slow (hours on CPU); turn on only if you have the time
import torch
HAS_GPU = torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0) if HAS_GPU else 'NONE - CPU ONLY')
REQUIRE_GPU = True                 # stages 03-08 need a GPU: on CPU, embeddings alone took ~6 h and CALF ~2.5 h
assert HAS_GPU or not REQUIRE_GPU or SMOKE, ("No GPU in this session. Settings -> Accelerator -> GPU T4 x2 "
                                             "(or your GPU quota is used up). Refusing to start a multi-hour CPU run.")
print('artifacts dir:', paths.ART, '| SMOKE =', SMOKE)

# FINAL RUN v1.2 (single notebook)
Runs stages 01-08 in order. Attach: causal-rag-kaggle dataset (code + all earlier results) and wordsforthewise/lending-club. GPU + Internet ON. Set SMOKE=True for a dry run first, then SMOKE=False and run as a Commit.

# 01 - Data preparation (real Lending Club loans)
Attach Kaggle dataset **wordsforthewise/lending-club**. Produces `lc_clean.parquet` + `censoring_report.csv`.
Leakage-free pre-origination fields only, no lender grade/interest rate, 36-month loans 2012-2015, resolved outcomes,
right-censoring guard, temporal split (train 2012-14, test 2015).

In [ ]:
from causal_rag import data_prep
import pandas as pd
if paths.have('lc_clean.parquet') and not SMOKE:
    print('stage 01 output exists -> skipping'); df = pd.read_parquet(paths.art('lc_clean.parquet'))
else:
    df = data_prep.prepare(smoke=SMOKE)
display(df.head())

# 02 - Structural causal graph
PC discovery under temporal-tier constraints + expert/mechanical edges. Outputs `graph.json`, `adjustment_sets.json`, `graph.png`.
Read the graph: **edit `EXPERT_EDGES` in `causal_rag/causal_graph.py`** if you disagree with an edge, then re-run.

In [ ]:
import pandas as pd, json
from IPython.display import Image, display
from causal_rag import causal_graph as cg
from causal_rag.config import TREATMENTS, fit_medians, add_features
df = pd.read_parquet(paths.art('lc_clean.parquet'))
tr = (df.role == 'train').values
d = add_features(df, fit_medians(df, tr))
g = cg.build_graph(d[tr]); adj = cg.adjustment_sets(g)
cg.draw_graph(g, paths.art('graph.png'), highlight=[s['var'] for s in TREATMENTS.values()])
display(Image(str(paths.art('graph.png'))))
print(g['method']); display(pd.DataFrame(g['edges'])[['src','dst','directed','kind']].head(60))
print(json.dumps(adj, indent=1))

# 03 - Real policy corpus + dense indexes
Downloads real documents (Fannie Mae Selling Guide DTI + credit score topics, CFPB Reg Z ability-to-repay, Reg B evaluation of applications),
chunks and entity-tags them, then embeds policy chunks and all labelled historical loan cards with `BAAI/bge-small-en-v1.5`.
Add your own real documents by attaching a dataset that contains a folder named `policy_docs/`.

In [ ]:
import pandas as pd
from causal_rag import policy_corpus as pc, pipeline as pl, embedding
if paths.have('policy_chunks.parquet','policy_emb.npy','case_emb.npy','case_cards.parquet'):
    print('stage 03 outputs already exist -> skipping (delete them to recompute)')
else:
    chunks = pc.build_chunks(fetch=True)
    display(chunks.head(3))
    emb = embedding.Embedder()
    pl.build_policy_index(emb, chunks)
    df = pd.read_parquet(paths.art('lc_clean.parquet'))
    pl.build_case_index(df, emb)

# 04 - Causal Attribution Agent (CALF on real data)
Per clause: CALF (layered, K=4 tokens) vs K=1 (paper-as-written attention), flat MLP, LightGBM T-learner, EconML CausalForestDML,
and an adjustment-set sensitivity run. No ground-truth ITE exists in real data: evaluation uses cross-fitted doubly-robust pseudo-outcomes
(DR-MSE, calibration slope, quintile tables). Saves `calf_scores.parquet`, `calf_models.pt`, `featurizers.pkl`, `calf_results.csv`.

In [ ]:
import pandas as pd, json
df = pd.read_parquet(paths.art('lc_clean.parquet'))
adj = json.load(open(paths.art('adjustment_sets.json')))
from causal_rag import pipeline as pl
if paths.have('calf_scores.parquet','calf_models.pt','featurizers.pkl','calf_results.csv'):
    print('stage 04 outputs already exist -> skipping (delete them to retrain)')
    res = pd.read_csv(paths.art('calf_results.csv'))
else:
    res, sc = pl.run_calf_stage(df, adj, seeds=(0,1,2), variant_seeds=(0,), epochs=EPOCHS, with_econml=WITH_ECONML)
display(res)

# 04b - Estimator validation on real RCTs + refutations on the observational effects
(a) Hillstrom (and Criteo if available): model ATE vs experimental diff-in-means, Qini, heterogeneity calibration.
(b) Placebo-treatment, random-common-cause, subset refuters and E-values for every clause effect on Lending Club.

In [ ]:
import numpy as np, pandas as pd
from causal_rag import rct_validation as rv
rows = []
try:
    h = rv.load_hillstrom(); L, T, y = rv.hillstrom_layers(h)
    if SMOKE: idx = np.random.default_rng(0).choice(len(y), 15000, replace=False); L=[x[idx] for x in L]; T=T[idx]; y=y[idx]
    out, last = rv.evaluate_rct(L, T, y, known_e=float(T.mean()), epochs=EPOCHS)
    out.insert(0, 'dataset', 'Hillstrom'); display(out); display(last[1]); rows.append(out)
except Exception as e:
    print('Hillstrom RCT validation SKIPPED (does not affect later stages):', type(e).__name__, str(e)[:400])
if not SMOKE:
    try:
        c = rv.load_criteo(); feats = [k for k in c.columns if k.startswith('f')]
        X = ((c[feats]-c[feats].mean())/c[feats].std()).values.astype('float32')
        o2, _ = rv.evaluate_rct([X], c.treatment.values.astype('float32'), c.visit.values.astype('float32'),
                                known_e=float(c.treatment.mean()), epochs=EPOCHS, flat_only=True)
        o2.insert(0, 'dataset', 'Criteo'); display(o2); rows.append(o2)
    except Exception as e: print('Criteo skipped:', type(e).__name__, e)
if rows: pd.concat(rows).to_csv(paths.art('rct_results.csv'), index=False)

In [ ]:
import json
from causal_rag import refute
from causal_rag.config import TREATMENTS, Featurizer, fit_medians, add_features
df = pd.read_parquet(paths.art('lc_clean.parquet'))
tr = (df.role=='train').values; te = (df.role=='test').values
d = add_features(df, fit_medians(df, tr)); yy = d.default.values
outs = []
for tk in TREATMENTS:
    fz = Featurizer(tk).fit(d, tr); X = np.hstack(fz.transform(d)); Tk = fz.treatment(d)
    outs.append(refute.run_refutations(X[te], Tk[te], yy[te], n_rep=1 if SMOKE else 3, max_n=8000 if SMOKE else 60000, name=tk))
R = pd.concat(outs); R.to_csv(paths.art('refutations.csv'), index=False)
display(refute.pass_rates(R)); display(R[R.test.isin(['estimate','e_value_point','e_value_ci_bound'])])

# 05 - Retriever Agent (dense retrieval + causal graph)
Evidence bundle per sampled 2015 applicant: matched treated-vs-control precedent per clause (balance-space matching on CALF scores),
clause-linked policy text (strict cutoff matching), vanilla / kNN baselines. Saves `evidence.pkl`, `eval_sample.parquet`.

In [ ]:
import json, pickle, numpy as np, pandas as pd
from causal_rag import retrieval as rt, embedding
from causal_rag.config import fit_medians, add_features, TREATMENTS
df = pd.read_parquet(paths.art('lc_clean.parquet'))
tr = (df.role=='train').values
medians = fit_medians(df, tr); d = add_features(df, medians)
idx = d[d.role.isin(['train','val'])].reset_index(drop=True)
assert (idx.case_id.values == pd.read_parquet(paths.art('case_cards.parquet')).case_id.values).all(), 'case index misaligned'
chunks = pd.read_parquet(paths.art('policy_chunks.parquet'))
emb = embedding.Embedder()
R = rt.RetrieverAgent(idx, np.load(paths.art('case_emb.npy')).astype('float32'), chunks,
                      np.load(paths.art('policy_emb.npy')).astype('float32'),
                      pd.read_parquet(paths.art('calf_scores.parquet')).set_index('case_id'), emb,
                      json.load(open(paths.art('adjustment_sets.json'))))
for tk in TREATMENTS:
    p = R.policy_for(tk)
    print(f"{tk:11s} policy-states-this-cutoff={p['supported']!s:5s} topical-only={p['topical']!s:5s} score={p['support_score']:.2f} ->",
          (p['chunks'][0]['publisher'] + ' | ' + p['chunks'][0]['title']) if p['chunks'] else 'NO POLICY TEXT (analyst-defined)')
test = d[d.role=='test'].sample(N_EVAL, random_state=0)       # same seed as earlier runs: the first 300 rows are the same applicants as v1.0/v1.1
bundles = R.build_bundles(test)
pickle.dump(bundles, open(paths.art('evidence.pkl'), 'wb')); test[['case_id','default']].to_parquet(paths.art('eval_sample.parquet'))
print('median balance SMD per clause:', {tk: round(float(np.nanmedian([x['evidence'][tk]['smd'] for x in bundles])),3) for tk in TREATMENTS})
print('thin-overlap share per clause :', {tk: round(float(np.mean([x['evidence'][tk]['thin'] for x in bundles])),3) for tk in TREATMENTS})

# 06 - LLM scoring + Counterfactual Perturbation Agent (resumable)
Seven systems are scored: no-RAG, vanilla RAG, kNN-RAG, Causal-RAG (v1.1 prompt), **Causal-RAG smooth** (new: all conditions always shown,
no appear/disappear cliff), and two control experiments (**CALF numbers only**, **matched precedent only**).
Counterfactual tests run on no-RAG, vanilla, Causal-RAG and Causal-RAG smooth. Progress is saved every `CHUNK` applicants in `llm_parts/`;
re-running the cell resumes where it stopped.

In [ ]:
import pickle, numpy as np, pandas as pd
from causal_rag import pipeline as pl
from causal_rag.llm import LLM
from causal_rag.config import fit_medians, add_features
bundles = pickle.load(open(paths.art('evidence.pkl'), 'rb'))
samp = pd.read_parquet(paths.art('eval_sample.parquet'))
df = pd.read_parquet(paths.art('lc_clean.parquet')); tr=(df.role=='train').values
d = add_features(df, fit_medians(df, tr)); idx = d[d.role.isin(['train','val'])].reset_index(drop=True)
llm = LLM()
BASE_SYSTEMS = ['no_rag','vanilla_rag','knn_rag','causal_rag','causal_rag_smooth','causal_calf_only','causal_precedent_only']
CF_SYSTEMS   = ['no_rag','vanilla_rag','causal_rag','causal_rag_smooth']
sc, cfdf = pl.run_llm_stage(llm, bundles, idx, BASE_SYSTEMS, CF_SYSTEMS, chunk=CHUNK)
sc = sc.merge(samp, on='case_id'); sc.to_parquet(paths.art('llm_scores.parquet')); cfdf.to_parquet(paths.art('cf_audit.parquet'))
display(cfdf.groupby(['system','kind']).size().unstack())

# 07 - Synthesis & Audit Agent
Deterministic audit rules over the counterfactual results of `AUDIT_SYSTEM` -> APPROVE / DECLINE / REFER_TO_HUMAN, causal DAG per case
(PNG + HTML), JSON record, LLM rationale (clearly separated from the audit table).

In [ ]:
import json, pickle, numpy as np, pandas as pd
from causal_rag import audit as au, causal_graph as cgm, pipeline as pl
from causal_rag.llm import LLM
bundles = pickle.load(open(paths.art('evidence.pkl'), 'rb'))
sc = pd.read_parquet(paths.art('llm_scores.parquet')); cfdf = pd.read_parquet(paths.art('cf_audit.parquet'))
g, adj = cgm.load_graph()
m = sc[AUDIT_SYSTEM].values
thr = float(np.quantile(m, 0.80)); band = float(0.15 * m.std())        # label-free operating point: decline the riskiest 20%
recs = [au.audit_case(b, cfdf[(cfdf.case_id==b['case_id'])&(cfdf.system==AUDIT_SYSTEM)], m[i], thr, band) for i,b in enumerate(bundles)]
os_ = paths.ART/'audits'; os_.mkdir(exist_ok=True)
llm = LLM() if N_REPORT else None
for i, r in enumerate(recs[:N_REPORT]):
    png = str(os_/f"case_{r['case_id']}.png"); au.render_dag(g, r, png)
    why = au.rationale(llm, bundles[i], r, AUDIT_SYSTEM); r['rationale'] = why
    (os_/f"case_{r['case_id']}.html").write_text(au.html_report(r, png, why))
with open(paths.art('decisions.jsonl'), 'w') as f:
    for r in recs: f.write(json.dumps(r, default=pl.json_default) + chr(10))
print('audit system:', AUDIT_SYSTEM); print(pd.Series([r['decision'] for r in recs]).value_counts())
print(pd.Series([fl.split(':')[0] for r in recs for fl in r['flags']]).value_counts())
from IPython.display import Image, display
if N_REPORT: display(Image(str(os_/f"case_{recs[0]['case_id']}.png"))); print(recs[0].get('rationale'))

# 08 - Evaluation, control experiments, comparison with all earlier runs, export
Discrimination + calibration, **paired control experiments** (is it the CALF numbers, the matched precedent, or the smoother layout?),
counterfactual consistency, audit coverage-risk, refutations, and a table across every earlier run stored in the dataset's `results/`.
The last cell exports this run so it can be added to `results/` for future comparisons.

In [ ]:
import json, pickle, numpy as np, pandas as pd, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from causal_rag import eval as ev, compare, refute
from causal_rag.config import Featurizer, fit_medians, add_features, TREATMENTS
sc = pd.read_parquet(paths.art('llm_scores.parquet')); cfdf = pd.read_parquet(paths.art('cf_audit.parquet'))
recs = [json.loads(l) for l in open(paths.art('decisions.jsonl'))]
y = sc.default.values
df = pd.read_parquet(paths.art('lc_clean.parquet')); tr=(df.role=='train').values; va=(df.role=='val').values
d = add_features(df, fit_medians(df, tr)); yy = d.default.values
flat = Featurizer(None).fit(d, tr); X = np.hstack(flat.transform(d))
te_idx = d.set_index('case_id').index.get_indexer(sc.case_id.values)
gbm = ev.gbm_baseline(X[tr], yy[tr], X[va], yy[va], X[te_idx])
calf = pd.read_parquet(paths.art('calf_scores.parquet')).set_index('case_id').loc[sc.case_id.values]
calf_risk = np.mean([calf[f'pf_{tk}'].values for tk in TREATMENTS], axis=0)
LLM_SYS = [c for c in sc.columns if c not in ('case_id','default')]
scores = {k: sc[k].values for k in LLM_SYS}
scores.update({'CALF factual risk (tabular)': calf_risk, 'LightGBM (tabular ceiling)': gbm})
B = 300 if SMOKE else 1000
disc = ev.discrimination_table(y, scores, ref='no_rag', B=B); display(disc)
if len(y) > 300:    # the first 300 applicants are the SAME ones scored in v1.0 / v1.1 (same seed) -> like-for-like view
    disc300 = ev.discrimination_table(y[:300], {k: v[:300] for k, v in scores.items()}, ref='no_rag', B=B)
    disc300.to_csv(paths.art('discrimination_first300.csv'), index=False); print('first 300 applicants only (comparable with earlier runs):'); display(disc300)
pairs = [('causal_rag_smooth','no_rag'), ('causal_rag','no_rag'), ('causal_rag_smooth','causal_rag'),
         ('causal_rag','causal_calf_only'), ('causal_rag','causal_precedent_only'),
         ('causal_calf_only','no_rag'), ('causal_precedent_only','no_rag'), ('causal_rag_smooth','knn_rag'), ('causal_rag_smooth','vanilla_rag')]
pw = ev.pairwise_table(y, scores, pairs, B=B); pw.to_csv(paths.art('pairwise_table.csv'), index=False); display(pw)
thr_by = {s: float(np.quantile(sc[s].values, 0.8)) for s in LLM_SYS}
cft = ev.counterfactual_table(cfdf, thr_by); cft.to_csv(paths.art('counterfactual_table.csv'), index=False); display(cft.T)
cov = ev.coverage_risk(y, [r['decision'] for r in recs]); json.dump(cov, open(paths.art('coverage_risk.json'),'w'), indent=1); print(cov)
ref = pd.read_csv(paths.art('refutations.csv')) if paths.have('refutations.csv') else None
if ref is not None: display(refute.pass_rates(ref))
disc.to_csv(paths.art('results_table.csv'), index=False)
json.dump(dict(run=RUN_NAME, audit_system=AUDIT_SYSTEM, discrimination=disc.to_dict('records'), counterfactual=cft.to_dict('records'),
               audit=cov, n_eval=len(y), default_rate=float(y.mean())), open(paths.art('metrics.json'), 'w'), indent=1)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
disc.set_index('system').auroc.plot.barh(ax=ax[0]); ax[0].set_title('AUROC (2015 hold-out sample)'); ax[0].axvline(.5, c='k', ls=':')
cft.set_index('system')[['sign_agree_all','sign_agree_responders','trivial_prior_agree']].plot.bar(ax=ax[1]); ax[1].set_title('flip direction agrees with CALF sign'); ax[1].axhline(.5, c='k', ls=':')
plt.tight_layout(); plt.savefig(paths.art('summary.png'), dpi=130); plt.show()
# ---- history across runs (uses the dataset's results/ folder) + export of this run
runs = compare.all_runs(RUN_NAME, disc, cft, cov)
print('earlier runs found:', list(compare.previous_runs()))
display(runs['discrimination'].pivot_table(index='system', columns='run', values='auroc').round(3))
display(runs['coverage'].set_index('run')[['n','approve_rate','decline_rate','refer_rate','default_rate_among_approved','default_rate_among_referred']].round(3))
out = compare.export_run(RUN_NAME, disc, cft, cov, dict(N_EVAL=N_EVAL, CHUNK=CHUNK, EPOCHS=EPOCHS, AUDIT_SYSTEM=AUDIT_SYSTEM, WITH_ECONML=WITH_ECONML, code_version='1.2.0'),
                         extra_files=[paths.art(n) for n in ('pairwise_table.csv','discrimination_first300.csv','calf_results.csv','rct_results.csv','refutations.csv','summary.png','metrics.json')])
print('exported to', out, '-> copy this folder into the dataset under results/ to keep the run')

### Optional: package this session's artifacts so they are never lost again
Download `crag_artifacts.zip` from the Output panel and upload it as a Kaggle Dataset (e.g. `crag-artifacts`). Attach that dataset in later notebooks: `restore_artifacts()` and the skip-if-exists guards pick everything up.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/crag_artifacts', 'zip', '/kaggle/working', 'artifacts')
print('wrote /kaggle/working/crag_artifacts.zip')